# Lab 1 · Serving + quantization — Kaggle T4 / P100 (interactive)

Runs the **serving** lab inside a Kaggle notebook session, which behaves like a
VM: a terminal, a GPU, and background processes. We start Ollama in the
background and benchmark `localhost` — same as on the 5090 node.

**Before you run:** Notebook settings → Accelerator = **GPU**, Internet = **On**
(account must be phone-verified).

**What it proves:** one model (Qwen2.5-3B) at **fp16 vs Q4** — Q4 uses ~half the
VRAM. Whether Q4 is also *faster* depends on the GPU: on a Blackwell 5090 yes
(~1.5×); on Kaggle's older P100/T4 the VRAM win holds but speed is about the same
(no int4/fp16 tensor acceleration). **Hardware decides whether quant buys speed.**

Ollama unloads an idle model after ~5 min, so the `nvidia-smi` check runs right
after each benchmark in the same cell to catch it loaded.

## Cell 1 — install & start Ollama in the background

In [ ]:
# Kaggle's image lacks zstd, which the Ollama installer now needs to extract.
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"])

# wait until the server answers before pulling (cold start can take >5s)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(2)
print("ollama up")

## Cell 2 — pull the same model at two precisions (the one network step)

In [ ]:
!ollama pull qwen2.5:3b                  # Q4_K_M, ~1.9 GB
!ollama pull qwen2.5:3b-instruct-fp16    # ~6.2 GB
!ollama list

## Cell 2b — live sanity check: one real chat completion

Confirm the model is actually serving on the OpenAI-compatible endpoint before we
benchmark — a single `curl` to `/v1/chat/completions`.

In [ ]:
!curl -s http://localhost:11434/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen2.5:3b","messages":[{"role":"user","content":"Say hello in one short sentence."}]}' \
  | python3 -c "import sys,json; d=json.load(sys.stdin); print('reply:', d['choices'][0]['message']['content']); print('usage:', d.get('usage'))"

## Cell 3 — write the benchmark script (identical to the repo's `bench.py`)

In [ ]:
%%writefile bench.py
#!/usr/bin/env python3
"""Tiny, dependency-free benchmark for any OpenAI-compatible chat endpoint."""
import argparse, json, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

PROMPT = "Write a clear 150-word explanation of what a GPU does in an AI system."

def call(base_url, model, max_tokens):
    body = json.dumps({"model": model,
        "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": max_tokens, "temperature": 0.7}).encode()
    req = urllib.request.Request(base_url.rstrip("/") + "/chat/completions",
        data=body, headers={"Content-Type": "application/json",
                            "Authorization": "Bearer none"})
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=300) as r:
        data = json.loads(r.read())
    dt = time.time() - t0
    usage = data.get("usage") or {}
    toks = usage.get("completion_tokens") or len(data["choices"][0]["message"]["content"].split())
    return dt, toks

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base-url", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--concurrency", type=int, default=16)
    ap.add_argument("--max-tokens", type=int, default=256)
    args = ap.parse_args()
    print(f"endpoint: {args.base_url}  model: {args.model}")
    print("warming up (loading model into VRAM)...")
    call(args.base_url, args.model, 8)
    dt, toks = call(args.base_url, args.model, args.max_tokens)
    print(f"\n[single]      latency {dt:5.2f}s  | {toks} tok | {toks/dt:6.1f} tok/s")
    n = args.concurrency
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n) as ex:
        results = list(ex.map(lambda _: call(args.base_url, args.model, args.max_tokens), range(n)))
    wall = time.time() - t0
    total_toks = sum(t for _, t in results)
    avg_lat = sum(d for d, _ in results) / n
    print(f"[concurrency {n:>2}] wall {wall:5.2f}s | {total_toks} tok total "
          f"| {total_toks/wall:6.1f} tok/s aggregate | avg req latency {avg_lat:5.2f}s")

if __name__ == "__main__":
    main()

## Cell 4 — benchmark Q4, then check VRAM

In [ ]:
!python bench.py --base-url http://localhost:11434/v1 --model qwen2.5:3b --concurrency 8
!nvidia-smi --query-gpu=memory.used --format=csv

## Cell 5 — benchmark fp16, then check VRAM

In [ ]:
!python bench.py --base-url http://localhost:11434/v1 --model qwen2.5:3b-instruct-fp16 --concurrency 8
!nvidia-smi --query-gpu=memory.used --format=csv